# Model Registry & Versioning — AI Lab Instructor Notebook

*MLOps & Systems · Expert*



7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import json
import hashlib
from pathlib import Path
import numpy as np
import time

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Model Registry & Versioning — Student Lab

Offline lab: implement a filesystem-backed model registry with versioning and stage promotion.

## 0 — Registry layout + helpers

In [ ]:
import json
import hashlib
from pathlib import Path
import numpy as np
import time

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Model Registry & Versioning — Instructor Lab

Offline lab: implement a filesystem-backed model registry with versioning and stage promotion.

## 0 — Registry layout + helpers

## Task 2: 0 — Registry layout + helpers

## 0 — Registry layout + helpers

In [ ]:
REGISTRY_ROOT = Path("registry")
MODEL_NAME = "toy_logreg"

def model_root(model_name: str) -> Path:
    return REGISTRY_ROOT / "models" / model_name

def versions_root(model_name: str) -> Path:
    return model_root(model_name) / "versions"

def stages_root(model_name: str) -> Path:
    return model_root(model_name) / "stages"

def ensure_layout(model_name: str):
    versions_root(model_name).mkdir(parents=True, exist_ok=True)
    stages_root(model_name).mkdir(parents=True, exist_ok=True)


## Task 3: 1 — Define a tiny model artifact + deterministic version id

## 1 — Define a tiny model artifact + deterministic version id

In [ ]:
def make_toy_model(seed: int = 0, d: int = 8):
    rng = np.random.default_rng(seed)
    w = rng.standard_normal((d,)).astype(np.float64)
    b = float(rng.standard_normal())
    return {"w": w, "b": b}

def stable_hash_bytes(parts: list[bytes]) -> str:
    h = hashlib.sha256()
    for p in parts:
        h.update(p)
    return h.hexdigest()[:16]

def version_id(model: dict, cfg: dict, metrics: dict) -> str:
    # stable: same (weights+cfg+metrics) => same id
    w_bytes = model["w"].tobytes()
    b_bytes = np.float64(model["b"]).tobytes()
    cfg_bytes = json.dumps(cfg, sort_keys=True).encode("utf-8")
    met_bytes = json.dumps(metrics, sort_keys=True).encode("utf-8")
    return stable_hash_bytes([w_bytes, b_bytes, cfg_bytes, met_bytes])

cfg = {"seed": 0, "d": 8}
metrics = {"acc": 0.82, "log_loss": 0.42}
model = make_toy_model(seed=cfg["seed"], d=cfg["d"])
vid = version_id(model, cfg, metrics)
print('version_id', vid)
check('vid_len', len(vid) == 16)


In [ ]:
# Checks
check('vid_len', len(vid) == 16)

## Task 4: 2 — Register a model version (artifacts + metadata)

## 2 — Register a model version (artifacts + metadata)

In [ ]:
$f

In [ ]:
# Checks
check('metadata_exists', (vdir1 / 'metadata.json').exists())

## Task 5: 3 — List versions + resolve stage pointers

## 3 — List versions + resolve stage pointers

In [ ]:
def load_metadata(model_name: str, vid: str) -> dict:
    p = versions_root(model_name) / vid / "metadata.json"
    return json.loads(p.read_text(encoding="utf-8"))

def list_versions(model_name: str):
    ensure_layout(model_name)
    out = []
    for v in sorted(versions_root(model_name).iterdir()):
        if not v.is_dir():
            continue
        meta = json.loads((v / "metadata.json").read_text(encoding="utf-8"))
        out.append(meta)
    return out

def stage_path(model_name: str, stage: str) -> Path:
    return stages_root(model_name) / f"{stage}.txt"

def get_stage(model_name: str, stage: str):
    p = stage_path(model_name, stage)
    return p.read_text(encoding="utf-8").strip() if p.exists() else None

vers = list_versions(MODEL_NAME)
print('n_versions', len(vers))
check('listed', len(vers) >= 1)
print('production pointer', get_stage(MODEL_NAME, 'production'))


In [ ]:
# Checks
check('listed', len(vers) >= 1)

## Task 6: 4 — Promote with gates (staging → production)

## 4 — Promote with gates (staging → production)

In [ ]:
$11

In [ ]:
# Checks
check('staging_points_good', get_stage(MODEL_NAME, 'staging') == vid1)
check('production_points_good', get_stage(MODEL_NAME, 'production') == vid1)

## Task 7: 5 — Audit: load production artifacts + compare versions

## 5 — Audit: load production artifacts + compare versions

In [ ]:
def load_artifacts(model_name: str, vid: str):
    vdir = versions_root(model_name) / vid
    w = np.load(vdir / "weights.npy")
    b = float((vdir / "bias.txt").read_text(encoding="utf-8").strip())
    meta = json.loads((vdir / "metadata.json").read_text(encoding="utf-8"))
    return {"w": w, "b": b}, meta

prod_vid = get_stage(MODEL_NAME, "production")
model_prod, meta_prod = load_artifacts(MODEL_NAME, prod_vid)
print('prod_vid', prod_vid)
print('prod_metrics', meta_prod['metrics'])
check('prod_acc_ok', float(meta_prod['metrics']['acc']) >= 0.80)

m1 = load_metadata(MODEL_NAME, vid1)
m2 = load_metadata(MODEL_NAME, vid2)
print('acc diff', float(m1['metrics']['acc']) - float(m2['metrics']['acc']))


In [ ]:
# Checks
check('prod_acc_ok', float(meta_prod['metrics']['acc']) >= 0.80)